# Schedules to Delta table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import sys
sys.path.insert(0, "/Workspace/Repos")
from transforms import conform_schedules

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/schedules/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/schedules/schedules"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/schedules"
TABLE      = "nfl.schedules"

In [0]:
schedule_schema = StructType([
    StructField("season",           IntegerType()),
    StructField("game_id",          StringType()),
    StructField("date",             StringType()),
    StructField("week",             IntegerType()),
    StructField("status",           StringType()),
    StructField("completed",        BooleanType()),
    StructField("home_team_id",     StringType()),
    StructField("home_team",        StringType()),
    StructField("home_score",       IntegerType()),
    StructField("away_team_id",     StringType()),
    StructField("away_team",        StringType()),
    StructField("away_score",       IntegerType()),
    StructField("venue",            StringType()),
])

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    season            INT       COMMENT 'NFL season year',
    game_id           STRING    COMMENT 'Game ID (primary key)',
    date              STRING    COMMENT 'Game date (ISO 8601)',
    week              INT       COMMENT 'NFL week number',
    status            STRING    COMMENT 'Game status (e.g., Final, In Progress)',
    completed         BOOLEAN   COMMENT 'Whether game has completed',
    home_team_id      STRING    COMMENT 'Home team ID',
    home_team         STRING    COMMENT 'Home team name',
    home_score        INT       COMMENT 'Home team score (null if not Final)',
    away_team_id      STRING    COMMENT 'Away team ID',
    away_team         STRING    COMMENT 'Away team name',
    away_score        INT       COMMENT 'Away team score (null if not Final)',
    venue             STRING    COMMENT 'Game venue',
    _source_file      STRING    COMMENT 'Blob file path',
    _ingested_at      TIMESTAMP COMMENT 'When ingested into Delta'
)
USING DELTA
LOCATION '{DEST}'
""")

In [0]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(schedule_schema)
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_snapshot_at"),
    ))

# Apply conform function, then add back metadata columns before writing stream
conformed = conform_schedules(raw)
schedules = (raw
    .select(
        *[F.col(c) for c in conformed.columns],  # All data columns from conform
        "_source_file",
        "_snapshot_at"
    )
    .withColumn("_ingested_at", F.current_timestamp())
)

In [0]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # Step 1: Dedupe within batch (keep newest snapshot per game)
    # Note: conform_schedules() strips schema to target columns, so use _ingested_at, not _snapshot_at
    w = Window.partitionBy("game_id").orderBy(F.col("_ingested_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))
    
    # Step 2: Auto-add missing columns (schema evolution)
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # Step 3: Merge into table
    (target_table.alias("t")
        .merge(latest.alias("s"), "t.game_id = s.game_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

(schedules.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT game_id) AS unique_games FROM nfl.schedules;

SELECT date, week, home_team, away_team, home_score, away_score, venue FROM nfl.schedules ORDER BY date DESC LIMIT 20;